3x3 2D XXZ experiment setup.


In [ ]:
import torch
import numpy as np
import quimb as qu
n=9

In [ ]:
def analytic(N, edge_full_data, node_full_data, num):
  analytical_energies = []  # analytical result
  analytical_states = []  # analytical result

  for k in range(num):
    H = np.zeros((2**N, 2**N), dtype=complex)  # NumPy zero matrix

    for (i, j), (J_xx, J_yy, J_zz) in edge_full_data[k].items():
      H += J_xx.item() * (
          qu.ikron(qu.pauli('X'), dims=[2]*N, inds=[i]) @
          qu.ikron(qu.pauli('X'), dims=[2]*N, inds=[j])
      )
      # YY term
      H += J_yy.item() * (
          qu.ikron(qu.pauli('Y'), dims=[2]*N, inds=[i]) @
          qu.ikron(qu.pauli('Y'), dims=[2]*N, inds=[j])
      )
      # ZZ term
      H += J_zz.item() * (
          qu.ikron(qu.pauli('Z'), dims=[2]*N, inds=[i]) @
          qu.ikron(qu.pauli('Z'), dims=[2]*N, inds=[j])
      )

    
    # single = np.array(node_full_data[k].detach().cpu().numpy())

    # for q in range(N):
    #   Kx, Ky, Kz = float(single[q, 1]), float(single[q, 2]), float(single[q, 3])
    #   H += Kx * qu.ikron(qu.pauli('X'), dims=[2]*N, inds=[q])
    #   H += Ky * qu.ikron(qu.pauli('Y'), dims=[2]*N, inds=[q])
    #   H += Kz * qu.ikron(qu.pauli('Z'), dims=[2]*N, inds=[q])

    analytical_energies.append(qu.linalg.base_linalg.groundenergy(H))
    analytical_states.append(qu.linalg.base_linalg.groundstate(H).reshape(-1))
  return analytical_energies, analytical_states

def mse(vector1, vector2):
    return np.mean((np.array(vector1) - np.array(vector2)) ** 2)


In [ ]:
train_edge_full_data = torch.load('edge_full_data_train.pt')
train_node_full_data = torch.load('node_full_data_train.pt')

In [ ]:
test_edge_full_data = torch.load('edge_full_data_test.pt')
test_node_full_data = torch.load('node_full_data_test.pt')


In [ ]:
analytical_energies_train, analytical_states_train =  analytic(n,train_edge_full_data, train_node_full_data, len(train_edge_full_data))
print("analytical_energies_train = ",analytical_energies_train)

analytical_energies_test, analytical_states_test =  analytic(n,test_edge_full_data, test_node_full_data, len(test_edge_full_data))
print("analytical_energies_test = ",analytical_energies_test)

In [ ]:
torch.save(analytical_energies_train, 'analytical_energies_train.pt')
torch.save(analytical_states_train, 'analytical_states_train.pt')
torch.save(analytical_energies_test, 'analytical_energies_test.pt')
torch.save(analytical_states_test, 'analytical_states_test.pt')